# Train LipNet-inspired Model for Real vs Fake Video Classification

This notebook trains a 3D Convolutional Neural Network (inspired by LipNet) to classify videos as REAL (0) or FAKE (1).
The model extracts a fixed number of frames from each video and uses 3D convolutions to capture spatiotemporal features.

In [1]:
# Import libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling3D, Dropout, Conv3D, MaxPooling3D, BatchNormalization, Activation, SpatialDropout3D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from pathlib import Path
from tqdm import tqdm
import glob

%matplotlib inline

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.13.0
GPU Available: []


## Step 1: Load and Prepare Dataset

In [2]:
# Configure dataset paths
DATASET_PATH = r'C:\Users\hp\Desktop\DeepTruth\data\raw\video'
REAL_VIDEOS_PATH = os.path.join(DATASET_PATH, 'DFD_original_sequences')
FAKE_VIDEOS_PATH = os.path.join(DATASET_PATH, 'DFD_manipulated_sequences')

# Hyperparameters for video processing
FRAMES_PER_VIDEO = 20
FRAME_HEIGHT = 100
FRAME_WIDTH = 100
CHANNELS = 3

print(f"Real videos path: {REAL_VIDEOS_PATH}")
print(f"Fake videos path: {FAKE_VIDEOS_PATH}")
print(f"Real path exists: {os.path.exists(REAL_VIDEOS_PATH)}")
print(f"Fake path exists: {os.path.exists(FAKE_VIDEOS_PATH)}")

def extract_frames(video_path, num_frames=FRAMES_PER_VIDEO, resize=(FRAME_HEIGHT, FRAME_WIDTH)):
    """
    Extracts a fixed number of frames from a video evenly spaced throughout the duration.
    """
    try:
        cap = cv2.VideoCapture(video_path)
        frames = []
        
        if not cap.isOpened():
            return None
            
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames == 0:
            cap.release()
            return None
        
        if total_frames < num_frames:
            frame_indices = np.arange(total_frames)
            padding = num_frames - total_frames
            frame_indices = np.pad(frame_indices, (0, padding), 'edge')
        else:
            frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
            
        for i in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (resize[1], resize[0]))
                frame = frame.astype('float32') / 255.0
                frames.append(frame)
            else:
                frames.append(np.zeros((resize[0], resize[1], 3), dtype=np.float32))
                
        cap.release()
        
        while len(frames) < num_frames:
            frames.append(np.zeros((resize[0], resize[1], 3), dtype=np.float32))
            
        return np.array(frames[:num_frames])
    except Exception as e:
        print(f"Error processing {video_path}: {str(e)}")
        return None

def load_videos_from_directory(directory_path, label, max_videos=None):
    """
    Load videos from directory and extract frames.
    """
    video_data = []
    labels = []
    
    if not os.path.exists(directory_path):
        print(f"Directory not found: {directory_path}")
        return np.array([]), np.array([])
    
    # Find all mp4 and avi files recursively
    video_files = glob.glob(os.path.join(directory_path, "**", "*.mp4"), recursive=True)
    video_files += glob.glob(os.path.join(directory_path, "**", "*.avi"), recursive=True)
    video_files += glob.glob(os.path.join(directory_path, "**", "*.mov"), recursive=True)
    video_files += glob.glob(os.path.join(directory_path, "**", "*.mkv"), recursive=True)
    
    print(f"Found {len(video_files)} video files in {os.path.basename(directory_path)}")
    
    if max_videos:
        video_files = video_files[:max_videos]
    
    successful_videos = 0
    failed_videos = 0
    
    for vid_path in tqdm(video_files, desc=f"Loading from {os.path.basename(directory_path)}"):
        try:
            frames = extract_frames(vid_path)
            if frames is not None:
                video_data.append(frames)
                labels.append(label)
                successful_videos += 1
            else:
                failed_videos += 1
        except Exception as e:
            failed_videos += 1
            continue
    
    print(f"Successfully loaded: {successful_videos}, Failed: {failed_videos}")
    
    if len(video_data) == 0:
        return np.array([]), np.array([])
    
    return np.array(video_data), np.array(labels)

print("\nLoading dataset...")

# Load real videos (label = 0)
print("\n--- Loading Real Videos ---")
X_real, y_real = load_videos_from_directory(REAL_VIDEOS_PATH, label=0, max_videos=250)

# Load fake videos (label = 1)
print("\n--- Loading Fake Videos ---")
X_fake, y_fake = load_videos_from_directory(FAKE_VIDEOS_PATH, label=1, max_videos=250)

print(f"\nReal videos shape: {X_real.shape}")
print(f"Fake videos shape: {X_fake.shape}")

# Combine datasets only if both have data
if len(X_real) > 0 and len(X_fake) > 0:
    X = np.concatenate([X_real, X_fake], axis=0)
    y = np.concatenate([y_real, y_fake], axis=0)
    
    print(f"\nDataset loaded:")
    print(f"  Total videos: {len(X)}")
    print(f"  Real videos (0): {np.sum(y == 0)}")
    print(f"  Fake videos (1): {np.sum(y == 1)}")
    print(f"  Video data shape: {X[0].shape} (frames, height, width, channels)")
else:
    print("ERROR: Could not load videos from one or both directories!")
    print(f"Real videos loaded: {len(X_real)}")
    print(f"Fake videos loaded: {len(X_fake)}")

Real videos path: C:\Users\hp\Desktop\DeepTruth\data\raw\video\DFD_original_sequences
Fake videos path: C:\Users\hp\Desktop\DeepTruth\data\raw\video\DFD_manipulated_sequences
Real path exists: True
Fake path exists: True

Loading dataset...

--- Loading Real Videos ---
Found 363 video files in DFD_original_sequences


Loading from DFD_original_sequences: 100%|██████████████| 250/250 [12:21<00:00,  2.97s/it]


Successfully loaded: 250, Failed: 0

--- Loading Fake Videos ---
Found 3068 video files in DFD_manipulated_sequences


Loading from DFD_manipulated_sequences: 100%|███████████| 250/250 [11:49<00:00,  2.84s/it]


Successfully loaded: 250, Failed: 0

Real videos shape: (250, 20, 100, 100, 3)
Fake videos shape: (250, 20, 100, 100, 3)

Dataset loaded:
  Total videos: 500
  Real videos (0): 250
  Fake videos (1): 250
  Video data shape: (20, 100, 100, 3) (frames, height, width, channels)


## Step 2: Split Data into Train/Test

In [3]:
# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} videos")
print(f"  Real: {np.sum(y_train == 0)}, Fake: {np.sum(y_train == 1)}")
print(f"\nTest set: {len(X_test)} videos")
print(f"  Real: {np.sum(y_test == 0)}, Fake: {np.sum(y_test == 1)}")

Training set: 400 videos
  Real: 200, Fake: 200

Test set: 100 videos
  Real: 50, Fake: 50


## Step 3: Build LipNet-inspired 3D CNN Model

In [4]:
def build_model(input_shape=(FRAMES_PER_VIDEO, FRAME_HEIGHT, FRAME_WIDTH, CHANNELS)):
    inputs = Input(shape=input_shape)
    
    # Spatiotemporal Convolution Block 1
    x = Conv3D(filters=32, kernel_size=(3, 5, 5), strides=(1, 2, 2), padding="same")(inputs)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = SpatialDropout3D(0.2)(x)
    x = MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2))(x)
    
    # Spatiotemporal Convolution Block 2
    x = Conv3D(filters=64, kernel_size=(3, 5, 5), strides=(1, 1, 1), padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = SpatialDropout3D(0.2)(x)
    x = MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2))(x)
    
    # Spatiotemporal Convolution Block 3
    x = Conv3D(filters=96, kernel_size=(3, 3, 3), strides=(1, 1, 1), padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = SpatialDropout3D(0.2)(x)
    x = MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2))(x)
    
    # Global Average Pooling to aggregate across time and space
    x = GlobalAveragePooling3D()(x)
    
    # Dense classification layers
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    predictions = Dense(1, activation='sigmoid')(x)  # Binary classification
    
    model = Model(inputs=inputs, outputs=predictions)
    
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

model = build_model()
print("Model created!")
print(f"Total parameters: {model.count_params():,}")
model.summary()

Model created!
Total parameters: 348,385
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 20, 100, 100, 3   0         
                             )]                                  
                                                                 
 conv3d (Conv3D)             (None, 20, 50, 50, 32)    7232      
                                                                 
 batch_normalization (Batch  (None, 20, 50, 50, 32)    128       
 Normalization)                                                  
                                                                 
 activation (Activation)     (None, 20, 50, 50, 32)    0         
                                                                 
 spatial_dropout3d (Spatial  (None, 20, 50, 50, 32)    0         
 Dropout3D)                                                      
                    

## Step 4: Train the Model

In [ ]:
# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

# Ensure models directory exists
os.makedirs(r'C:\Users\hp\Desktop\DeepTruth\models', exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath=r'C:\Users\hp\Desktop\DeepTruth\models\lipnet_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Train model
print("Training model...")
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=8,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)

Training model...
Epoch 1/30
45/45 [==============================] - ETA: 0s - loss: 0.7966 - accuracy: 0.4667
Epoch 1: val_accuracy improved from -inf to 0.50000, saving model to C:\Users\hp\Desktop\DeepTruth\models\lipnet_best_model.h5
45/45 [==============================] - 140s 3s/step - loss: 0.7966 - accuracy: 0.4667 - val_loss: 0.6947 - val_accuracy: 0.5000 - lr: 1.0000e-04
Epoch 2/30


C:\Users\hp\Desktop\DeepTruth\venv\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


38/45 [========================>.....] - ETA: 21s - loss: 0.8076 - accuracy: 0.4408

## Step 5: Evaluate Model

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Get predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Calculate metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))

print("Detailed Metrics:")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")

## Step 6: Plot Training History

In [ ]:
# Plot accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r'C:\Users\hp\Desktop\DeepTruth\models\lipnet_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training history saved to C:\\Users\\hp\\Desktop\\DeepTruth\\models\\lipnet_training_history.png")

## Step 7: Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(r'C:\Users\hp\Desktop\DeepTruth\models\lipnet_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to C:\\Users\\hp\\Desktop\\DeepTruth\\models\\lipnet_confusion_matrix.png")

## Step 8: Save the Trained Model

In [ ]:
from pathlib import Path

print(f"CWD: {Path.cwd()}")

dest_dir = Path(r"C:\Users\hp\Desktop\DeepTruth\models")
dest_dir.mkdir(parents=True, exist_ok=True)

model_path = dest_dir / "lipnet_deepfake_classifier.h5"
model.save(model_path)

if not model_path.exists():
    raise FileNotFoundError(f"Model save reported success but file is missing: {model_path}")

model_bytes = model_path.stat().st_size
if model_bytes == 0:
    raise OSError(f"Model file was created but is empty: {model_path}")

print(f"Model saved to: {model_path.resolve()} ({model_bytes} bytes)")

weights_path = dest_dir / "lipnet_deepfake_weights.h5"
model.save_weights(weights_path)

if not weights_path.exists():
    raise FileNotFoundError(f"Weights save reported success but file is missing: {weights_path}")

weights_bytes = weights_path.stat().st_size
if weights_bytes == 0:
    raise OSError(f"Weights file was created but is empty: {weights_path}")

print(f"Weights saved to: {weights_path.resolve()} ({weights_bytes} bytes)")

## Step 9: Test the Model on Individual Videos

In [ ]:
def predict_deepfake_video(video_path, model):
    """
    Predict if a video is real or fake.
    """
    # Load and preprocess video
    frames = extract_frames(video_path)
    
    if frames is None:
        return None
    
    # Add batch dimension
    frames = np.expand_dims(frames, axis=0)
    
    # Predict
    prediction = model.predict(frames, verbose=0)[0][0]
    
    # Format result
    if prediction > 0.5:
        label = 'FAKE'
        confidence = prediction
    else:
        label = 'REAL'
        confidence = 1 - prediction
    
    return {
        'label': label,
        'confidence': float(confidence),
        'raw_prediction': float(prediction)
    }

print("Prediction function defined.")

## Step 10: Test on Sample Videos

In [ ]:
import glob

# Find some real videos to test
real_test_files = glob.glob(os.path.join(REAL_VIDEOS_PATH, "**", "*.mp4"), recursive=True)
real_test_files += glob.glob(os.path.join(REAL_VIDEOS_PATH, "**", "*.avi"), recursive=True)
    
print("=== Testing on Real Videos ===")
for vid_path in real_test_files[:3]:
    vid_name = os.path.basename(vid_path)
    result = predict_deepfake_video(vid_path, model)
    if result:
        print(f"{vid_name}: {result['label']} ({result['confidence']:.2%})")

# Find some fake videos to test
fake_test_files = glob.glob(os.path.join(FAKE_VIDEOS_PATH, "**", "*.mp4"), recursive=True)
fake_test_files += glob.glob(os.path.join(FAKE_VIDEOS_PATH, "**", "*.avi"), recursive=True)
    
print("\n=== Testing on Fake Videos ===")
for vid_path in fake_test_files[:3]:
    vid_name = os.path.basename(vid_path)
    result = predict_deepfake_video(vid_path, model)
    if result:
        print(f"{vid_name}: {result['label']} ({result['confidence']:.2%})")